In [ ]:
%matplotlib inline

from collections import defaultdict, Counter
import urllib

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn import preprocessing
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import fcluster

import nimfa
import fastcluster

# H3K27ac

In [ ]:
data = pd.read_table('../matrix/H3K27ac_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)

In [ ]:
data.columns = data.columns.str[:-5]
data_avg = data.groupby(data.columns, axis=1).mean()

In [ ]:
data = np.array(data_avg).T

In [ ]:
# level of sparsity
print(np.sum(data != 0)/data.size)

# negative elements
print(np.sum(data < 0))

In [ ]:
data.shape

In [ ]:
# normalize data
data = preprocessing.Normalizer().fit_transform(data)

In [ ]:
data.to_csv('./tutorial_data.csv')

In [ ]:
rank_cands = range(5, 20, 2)
snmf = nimfa.Nmf(data, seed='nndsvd', max_iter=100)
summary = snmf.estimate_rank(rank_range=rank_cands, n_run=10, what='all')

In [ ]:
rss = [summary[rank]['rss'] for rank in rank_cands]
coph = [summary[rank]['cophenetic'] for rank in rank_cands]
disp = [summary[rank]['dispersion'] for rank in rank_cands]
spar = [summary[rank]['sparseness'] for rank in rank_cands]
spar_w, spar_h = zip(*spar)
evar = [summary[rank]['evar'] for rank in rank_cands]

# 创建多个子图
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
fig.tight_layout(pad=5.0)

# 绘制 RSS
axes[0, 0].plot(rank_cands, rss, 'o-', color='blue', linewidth=2)
axes[0, 0].set_title('RSS')
axes[0, 0].set_xlabel('Rank')
axes[0, 0].set_ylabel('RSS Value')
axes[0, 0].grid(True)

# 绘制 Cophenetic correlation
axes[0, 1].plot(rank_cands, coph, 'o-', color='green', linewidth=2)
axes[0, 1].set_title('Cophenetic Correlation')
axes[0, 1].set_xlabel('Rank')
axes[0, 1].set_ylabel('Cophenetic Value')
axes[0, 1].grid(True)

# 绘制 Dispersion
axes[1, 0].plot(rank_cands, disp, 'o-', color='red', linewidth=2)
axes[1, 0].set_title('Dispersion')
axes[1, 0].set_xlabel('Rank')
axes[1, 0].set_ylabel('Dispersion Value')
axes[1, 0].grid(True)

# 绘制 Sparsity (Basis)
axes[1, 1].plot(rank_cands, spar_w, 'o-', color='purple', linewidth=2)
axes[1, 1].set_title('Sparsity (Basis)')
axes[1, 1].set_xlabel('Rank')
axes[1, 1].set_ylabel('Sparsity Value')
axes[1, 1].grid(True)

# 绘制 Sparsity (Mixture)
axes[2, 0].plot(rank_cands, spar_h, 'o-', color='orange', linewidth=2)
axes[2, 0].set_title('Sparsity (Mixture)')
axes[2, 0].set_xlabel('Rank')
axes[2, 0].set_ylabel('Sparsity Value')
axes[2, 0].grid(True)

# 绘制 Explained variance
axes[2, 1].plot(rank_cands, evar, 'o-', color='brown', linewidth=2)
axes[2, 1].set_title('Explained Variance')
axes[2, 1].set_xlabel('Rank')
axes[2, 1].set_ylabel('Explained Variance Value')
axes[2, 1].grid(True)

# 显示图像
plt.show()

'''
plt.plot(rank_cands, rss, 'o-', label='RSS', linewidth=2)
plt.plot(rank_cands, coph, 'o-', label='Cophenetic correlation', linewidth=2)
plt.plot(rank_cands, disp,'o-', label='Dispersion', linewidth=2)
plt.plot(rank_cands, spar_w, 'o-', label='Sparsity (Basis)', linewidth=2)
plt.plot(rank_cands, spar_h, 'o-', label='Sparsity (Mixture)', linewidth=2)
plt.plot(rank_cands, evar, 'o-', label='Explained variance', linewidth=2)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, numpoints=1);
'''

## Clustering of genes

In [ ]:
rank = 9
nmf = nimfa.Nmf(data.T, rank=rank, seed='nndsvd', max_iter=100, 
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

### plot dendrogram along with the heatmap

In [ ]:
C = 1 - nmf_fit.fit.consensus()
Y = fastcluster.linkage(C, method='average')

In [ ]:
import sys
sys.setrecursionlimit(100000)

In [ ]:
D.shape

In [ ]:
def clean_axis(ax):
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

#Y = sch.linkage(C, method='average')
#Y = fastcluster.linkage(C, method='average')

denD = sch.dendrogram(Y, orientation='left', link_color_func=lambda k: 'black')

D = C[denD['leaves'], :][:, denD['leaves']]


In [ ]:
fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])
denAX = fig.add_subplot(heatmapGS[0,0])
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
axi = heatmapAX.imshow(D, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)
plt.savefig("t/H3K27ac/consensus_heatmap.pdf", dpi = 300, bbox_inches='tight')
plt.show()

In [ ]:
# Assign data points to clusters using fcluster
num_clusters = 9  # Set the desired number of clusters
clusters = sch.fcluster(Y, t=num_clusters, criterion='maxclust')

# Combine data points and their cluster identities
data_with_clusters = np.column_stack((data_avg.index, clusters))

# Save cluster assignments to a text file
np.savetxt('t/H3K27ac/cluster_assignments.txt', data_with_clusters, fmt='%s', delimiter='\t', 
           header='Peak\tCluster', comments='')

In [ ]:
pd.Series(clusters).value_counts().sort_index()

In [ ]:
W = nmf_fit.fit.basis()

c = 0
k = 10

topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
val = W[topk, c]

plt.barh(np.arange(k) + .5, val.A1, align="center")
labels = [data_avg.index[idx] for idx in topk]
plt.yticks(np.arange(k) + .5, labels)
plt.xlabel("Weight")
plt.ylabel("Gene");

In [ ]:
data_raw = pd.read_table('../matrix/H3K27ac_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)
exp_df = pd.DataFrame()
for c in range(9):
    k = 50
    
    topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
    val = W[topk, c]
    
    
    labels = [data_avg.index[idx] for idx in topk]

    tmp = data_raw.loc[labels,:]
    tmp.index = str(c) + '-' + tmp.index
    exp_df = pd.concat([exp_df, tmp], axis=0, ignore_index=False)
    
    plt.figure(figsize=(18, 10))
    plt.barh(np.arange(k) + .5, val.A1, align="center")
    plt.yticks(np.arange(k) + .5, labels, fontsize=8)
    plt.xlabel("Weight")
    plt.ylabel("Position")
    plt.savefig("result/H3K27ac/top50_weighted_position_" + str(c) + ".pdf", dpi = 300)
    plt.close() 

In [ ]:
exp_df.to_csv('result/H3K27ac/top50_weighted_position_level.tsv', sep = '\t')

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(exp_df, annot=False, cmap='coolwarm', yticklabels=False)
plt.savefig("result/H3K27ac/top50_weighted_position_level_heatmap.pdf", dpi = 300, bbox_inches='tight')
plt.close()

# H3K9me3

In [ ]:
data_raw = pd.read_table('../matrix/H3K9me3_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)

In [ ]:
data_raw.columns = data_raw.columns.str[:-5]
data_avg = data_raw.groupby(data_raw.columns, axis=1).mean()

In [ ]:
data = np.array(data_avg)

In [ ]:
# level of sparsity
print(np.sum(data != 0)/data.size)

# negative elements
print(np.sum(data < 0))

In [ ]:
data.shape

In [ ]:
# normalize data
data = preprocessing.Normalizer().fit_transform(data)

In [ ]:
rank_cands = range(5, 16, 1)
nmf = nimfa.Nmf(data.T, seed='nndsvd', max_iter=100)
summary = nmf.estimate_rank(rank_range=rank_cands, n_run=10, what='all')

In [ ]:
rss = [summary[rank]['rss'] for rank in rank_cands]
coph = [summary[rank]['cophenetic'] for rank in rank_cands]
disp = [summary[rank]['dispersion'] for rank in rank_cands]
spar = [summary[rank]['sparseness'] for rank in rank_cands]
spar_w, spar_h = zip(*spar)
evar = [summary[rank]['evar'] for rank in rank_cands]

# 创建多个子图
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
fig.tight_layout(pad=5.0)

# 绘制 RSS
axes[0, 0].plot(rank_cands, rss, 'o-', color='blue', linewidth=2)
axes[0, 0].set_title('RSS')
axes[0, 0].set_xlabel('Rank')
axes[0, 0].set_ylabel('RSS Value')
axes[0, 0].grid(True)

# 绘制 Cophenetic correlation
axes[0, 1].plot(rank_cands, coph, 'o-', color='green', linewidth=2)
axes[0, 1].set_title('Cophenetic Correlation')
axes[0, 1].set_xlabel('Rank')
axes[0, 1].set_ylabel('Cophenetic Value')
axes[0, 1].grid(True)

# 绘制 Dispersion
axes[1, 0].plot(rank_cands, disp, 'o-', color='red', linewidth=2)
axes[1, 0].set_title('Dispersion')
axes[1, 0].set_xlabel('Rank')
axes[1, 0].set_ylabel('Dispersion Value')
axes[1, 0].grid(True)

# 绘制 Sparsity (Basis)
axes[1, 1].plot(rank_cands, spar_w, 'o-', color='purple', linewidth=2)
axes[1, 1].set_title('Sparsity (Basis)')
axes[1, 1].set_xlabel('Rank')
axes[1, 1].set_ylabel('Sparsity Value')
axes[1, 1].grid(True)

# 绘制 Sparsity (Mixture)
axes[2, 0].plot(rank_cands, spar_h, 'o-', color='orange', linewidth=2)
axes[2, 0].set_title('Sparsity (Mixture)')
axes[2, 0].set_xlabel('Rank')
axes[2, 0].set_ylabel('Sparsity Value')
axes[2, 0].grid(True)

# 绘制 Explained variance
axes[2, 1].plot(rank_cands, evar, 'o-', color='brown', linewidth=2)
axes[2, 1].set_title('Explained Variance')
axes[2, 1].set_xlabel('Rank')
axes[2, 1].set_ylabel('Explained Variance Value')
axes[2, 1].grid(True)

# 显示图像
plt.show()

'''
plt.plot(rank_cands, rss, 'o-', label='RSS', linewidth=2)
plt.plot(rank_cands, coph, 'o-', label='Cophenetic correlation', linewidth=2)
plt.plot(rank_cands, disp,'o-', label='Dispersion', linewidth=2)
plt.plot(rank_cands, spar_w, 'o-', label='Sparsity (Basis)', linewidth=2)
plt.plot(rank_cands, spar_h, 'o-', label='Sparsity (Mixture)', linewidth=2)
plt.plot(rank_cands, evar, 'o-', label='Explained variance', linewidth=2)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, numpoints=1);
'''

## Clustering of genes

In [ ]:
rank = 10
nmf = nimfa.Nmf(data, rank=rank, seed='nndsvd', random_state=123, max_iter=100, 
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

In [ ]:
rank = 10
nmf = nimfa.Nmf(data.T, rank=rank, seed='nndsvd', random_state=123, max_iter=100, 
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

### plot dendrogram along with the heatmap

In [ ]:
def clean_axis(ax):
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C = 1 - nmf_fit.fit.consensus()
Y = sch.linkage(C, method='average')

denAX = fig.add_subplot(heatmapGS[0,0])
denD = sch.dendrogram(Y, orientation='left', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D = C[denD['leaves'], :][:, denD['leaves']]
axi = heatmapAX.imshow(D, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

In [ ]:
plt.savefig("t/result/H3K9me3/top50_weighted_position_level_heatmap.pdf", dpi = 300, bbox_inches='tight')

In [ ]:
W = nmf_fit.fit.basis()

c = 0
k = 10

topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
val = W[topk, c]

plt.barh(np.arange(k) + .5, val.A1, align="center")
labels = [data_avg.index[idx] for idx in topk]
plt.yticks(np.arange(k) + .5, labels)
plt.xlabel("Weight")
plt.ylabel("Gene");

In [ ]:
data_raw = pd.read_table('../matrix/H3K9me3_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)
exp_df = pd.DataFrame()
for c in range(10):
    k = 50
    
    topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
    val = W[topk, c]
    
    
    labels = [data_avg.index[idx] for idx in topk]

    tmp = data_raw.loc[labels,:]
    tmp.index = str(c) + '-' + tmp.index
    exp_df = pd.concat([exp_df, tmp], axis=0, ignore_index=False)
    
    plt.figure(figsize=(18, 10))
    plt.barh(np.arange(k) + .5, val.A1, align="center")
    plt.yticks(np.arange(k) + .5, labels, fontsize=8)
    plt.xlabel("Weight")
    plt.ylabel("Position")
    plt.savefig("result/H3K9me3/top50_weighted_position_" + str(c) + ".pdf", dpi = 300)
    plt.close() 

In [ ]:
exp_df.to_csv('result/H3K9me3/top50_weighted_position_level.tsv', sep = '\t')

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(exp_df, annot=False, cmap='coolwarm', yticklabels=False)
plt.savefig("result/H3K9me3/top50_weighted_position_level_heatmap.pdf", dpi = 300, bbox_inches='tight')
plt.close()

# H3K4me3

In [ ]:
data_raw = pd.read_table('../matrix/H3K4me3_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)

In [ ]:
data_raw.columns = data_raw.columns.str[:-5]
data_avg = data_raw.groupby(data_raw.columns, axis=1).mean()

In [ ]:
data = np.array(data_avg)

In [ ]:
# level of sparsity
print(np.sum(data != 0)/data.size)

# negative elements
print(np.sum(data < 0))

In [ ]:
# normalize data
#data = preprocessing.Normalizer().fit_transform(data)

In [ ]:
rank_cands = range(5, 16, 1)
nmf = nimfa.Nmf(data, seed='nndsvd', max_iter=100, random_state = 123)
summary = nmf.estimate_rank(rank_range=rank_cands, n_run=10, what='all')

In [ ]:
rss = [summary[rank]['rss'] for rank in rank_cands]
coph = [summary[rank]['cophenetic'] for rank in rank_cands]
disp = [summary[rank]['dispersion'] for rank in rank_cands]
spar = [summary[rank]['sparseness'] for rank in rank_cands]
spar_w, spar_h = zip(*spar)
evar = [summary[rank]['evar'] for rank in rank_cands]

# 创建多个子图
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
fig.tight_layout(pad=5.0)

# 绘制 RSS
axes[0, 0].plot(rank_cands, rss, 'o-', color='blue', linewidth=2)
axes[0, 0].set_title('RSS')
axes[0, 0].set_xlabel('Rank')
axes[0, 0].set_ylabel('RSS Value')
axes[0, 0].grid(True)

# 绘制 Cophenetic correlation
axes[0, 1].plot(rank_cands, coph, 'o-', color='green', linewidth=2)
axes[0, 1].set_title('Cophenetic Correlation')
axes[0, 1].set_xlabel('Rank')
axes[0, 1].set_ylabel('Cophenetic Value')
axes[0, 1].grid(True)

# 绘制 Dispersion
axes[1, 0].plot(rank_cands, disp, 'o-', color='red', linewidth=2)
axes[1, 0].set_title('Dispersion')
axes[1, 0].set_xlabel('Rank')
axes[1, 0].set_ylabel('Dispersion Value')
axes[1, 0].grid(True)

# 绘制 Sparsity (Basis)
axes[1, 1].plot(rank_cands, spar_w, 'o-', color='purple', linewidth=2)
axes[1, 1].set_title('Sparsity (Basis)')
axes[1, 1].set_xlabel('Rank')
axes[1, 1].set_ylabel('Sparsity Value')
axes[1, 1].grid(True)

# 绘制 Sparsity (Mixture)
axes[2, 0].plot(rank_cands, spar_h, 'o-', color='orange', linewidth=2)
axes[2, 0].set_title('Sparsity (Mixture)')
axes[2, 0].set_xlabel('Rank')
axes[2, 0].set_ylabel('Sparsity Value')
axes[2, 0].grid(True)

# 绘制 Explained variance
axes[2, 1].plot(rank_cands, evar, 'o-', color='brown', linewidth=2)
axes[2, 1].set_title('Explained Variance')
axes[2, 1].set_xlabel('Rank')
axes[2, 1].set_ylabel('Explained Variance Value')
axes[2, 1].grid(True)

# 显示图像
plt.show()

'''
plt.plot(rank_cands, rss, 'o-', label='RSS', linewidth=2)
plt.plot(rank_cands, coph, 'o-', label='Cophenetic correlation', linewidth=2)
plt.plot(rank_cands, disp,'o-', label='Dispersion', linewidth=2)
plt.plot(rank_cands, spar_w, 'o-', label='Sparsity (Basis)', linewidth=2)
plt.plot(rank_cands, spar_h, 'o-', label='Sparsity (Mixture)', linewidth=2)
plt.plot(rank_cands, evar, 'o-', label='Explained variance', linewidth=2)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, numpoints=1);
'''

## Clustering of genes

In [ ]:
rank = 10
nmf = nimfa.Nmf(data.T, rank=rank, seed='nndsvd', max_iter=100, 
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

In [ ]:
plt.savefig("t/result/H3K4me3/top50_weighted_position_level_heatmap.pdf", dpi = 300, bbox_inches='tight')

### plot dendrogram along with the heatmap

In [ ]:
def clean_axis(ax):
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C = 1 - nmf_fit.fit.consensus()
Y = sch.linkage(C, method='average')

denAX = fig.add_subplot(heatmapGS[0,0])
denD = sch.dendrogram(Y, orientation='left', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D = C[denD['leaves'], :][:, denD['leaves']]
axi = heatmapAX.imshow(D, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

In [ ]:
W = nmf_fit.fit.basis()

c = 0
k = 10

topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
val = W[topk, c]

plt.barh(np.arange(k) + .5, val.A1, align="center")
labels = [data_avg.index[idx] for idx in topk]
plt.yticks(np.arange(k) + .5, labels)
plt.xlabel("Weight")
plt.ylabel("Gene");

In [ ]:
data_raw = pd.read_table('../matrix/H3K4me3_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)
exp_df = pd.DataFrame()
for c in range(10):
    k = 50
    
    topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
    val = W[topk, c]
    
    
    labels = [data_avg.index[idx] for idx in topk]

    tmp = data_raw.loc[labels,:]
    tmp.index = str(c) + '-' + tmp.index
    exp_df = pd.concat([exp_df, tmp], axis=0, ignore_index=False)
    
    plt.figure(figsize=(18, 10))
    plt.barh(np.arange(k) + .5, val.A1, align="center")
    plt.yticks(np.arange(k) + .5, labels, fontsize=8)
    plt.xlabel("Weight")
    plt.ylabel("Position")
    plt.savefig("result/H3K4me3/top50_weighted_position_" + str(c) + ".pdf", dpi = 300)
    plt.close() 

In [ ]:
exp_df.to_csv('result/H3K4me3/top50_weighted_position_level.tsv', sep = '\t')

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(exp_df, annot=False, cmap='coolwarm', yticklabels=False)
plt.savefig("result/H3K4me3/top50_weighted_position_level_heatmap.pdf", dpi = 300, bbox_inches='tight')
plt.close()

# H3K4me1

In [ ]:
data_raw = pd.read_table('../matrix/H3K4me1_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)

In [ ]:
data_raw.columns = data_raw.columns.str[:-5]
data_avg = data_raw.groupby(data_raw.columns, axis=1).mean()

In [ ]:
data = np.array(data_avg)

In [ ]:
# level of sparsity
print(np.sum(data != 0)/data.size)

# negative elements
print(np.sum(data < 0))

In [ ]:
# normalize data
#data = preprocessing.Normalizer().fit_transform(data)

In [ ]:
rank_cands = range(5, 16, 1)
nmf = nimfa.Nmf(data, seed='nndsvd', max_iter=100)
summary = nmf.estimate_rank(rank_range=rank_cands, n_run=10, what='all')

In [ ]:
rss = [summary[rank]['rss'] for rank in rank_cands]
coph = [summary[rank]['cophenetic'] for rank in rank_cands]
disp = [summary[rank]['dispersion'] for rank in rank_cands]
spar = [summary[rank]['sparseness'] for rank in rank_cands]
spar_w, spar_h = zip(*spar)
evar = [summary[rank]['evar'] for rank in rank_cands]

# 创建多个子图
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
fig.tight_layout(pad=5.0)

# 绘制 RSS
axes[0, 0].plot(rank_cands, rss, 'o-', color='blue', linewidth=2)
axes[0, 0].set_title('RSS')
axes[0, 0].set_xlabel('Rank')
axes[0, 0].set_ylabel('RSS Value')
axes[0, 0].grid(True)

# 绘制 Cophenetic correlation
axes[0, 1].plot(rank_cands, coph, 'o-', color='green', linewidth=2)
axes[0, 1].set_title('Cophenetic Correlation')
axes[0, 1].set_xlabel('Rank')
axes[0, 1].set_ylabel('Cophenetic Value')
axes[0, 1].grid(True)

# 绘制 Dispersion
axes[1, 0].plot(rank_cands, disp, 'o-', color='red', linewidth=2)
axes[1, 0].set_title('Dispersion')
axes[1, 0].set_xlabel('Rank')
axes[1, 0].set_ylabel('Dispersion Value')
axes[1, 0].grid(True)

# 绘制 Sparsity (Basis)
axes[1, 1].plot(rank_cands, spar_w, 'o-', color='purple', linewidth=2)
axes[1, 1].set_title('Sparsity (Basis)')
axes[1, 1].set_xlabel('Rank')
axes[1, 1].set_ylabel('Sparsity Value')
axes[1, 1].grid(True)

# 绘制 Sparsity (Mixture)
axes[2, 0].plot(rank_cands, spar_h, 'o-', color='orange', linewidth=2)
axes[2, 0].set_title('Sparsity (Mixture)')
axes[2, 0].set_xlabel('Rank')
axes[2, 0].set_ylabel('Sparsity Value')
axes[2, 0].grid(True)

# 绘制 Explained variance
axes[2, 1].plot(rank_cands, evar, 'o-', color='brown', linewidth=2)
axes[2, 1].set_title('Explained Variance')
axes[2, 1].set_xlabel('Rank')
axes[2, 1].set_ylabel('Explained Variance Value')
axes[2, 1].grid(True)

# 显示图像
plt.show()

'''
plt.plot(rank_cands, rss, 'o-', label='RSS', linewidth=2)
plt.plot(rank_cands, coph, 'o-', label='Cophenetic correlation', linewidth=2)
plt.plot(rank_cands, disp,'o-', label='Dispersion', linewidth=2)
plt.plot(rank_cands, spar_w, 'o-', label='Sparsity (Basis)', linewidth=2)
plt.plot(rank_cands, spar_h, 'o-', label='Sparsity (Mixture)', linewidth=2)
plt.plot(rank_cands, evar, 'o-', label='Explained variance', linewidth=2)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, numpoints=1);
'''

## Clustering of genes

In [ ]:
rank = 11
nmf = nimfa.Nmf(data.T, rank=rank, seed='nndsvd', max_iter=100, random_state = 123,
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

### plot dendrogram along with the heatmap

In [ ]:
def clean_axis(ax):
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C = 1 - nmf_fit.fit.consensus()
Y = sch.linkage(C, method='average')

denAX = fig.add_subplot(heatmapGS[0,0])
denD = sch.dendrogram(Y, orientation='left', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D = C[denD['leaves'], :][:, denD['leaves']]
axi = heatmapAX.imshow(D, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

In [ ]:
plt.savefig("t/result/H3K4me1/top50_weighted_position_level_heatmap.pdf", dpi = 300, bbox_inches='tight')

In [ ]:
W = nmf_fit.fit.basis()

c = 0
k = 10

topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
val = W[topk, c]

plt.barh(np.arange(k) + .5, val.A1, align="center")
labels = [data_avg.index[idx] for idx in topk]
plt.yticks(np.arange(k) + .5, labels)
plt.xlabel("Weight")
plt.ylabel("Gene");

In [ ]:
data_raw = pd.read_table('../matrix/H3K4me1_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)
exp_df = pd.DataFrame()
for c in range(11):
    k = 50
    
    topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
    val = W[topk, c]
    
    
    labels = [data_avg.index[idx] for idx in topk]

    tmp = data_raw.loc[labels,:]
    tmp.index = str(c) + '-' + tmp.index
    exp_df = pd.concat([exp_df, tmp], axis=0, ignore_index=False)
    
    plt.figure(figsize=(18, 10))
    plt.barh(np.arange(k) + .5, val.A1, align="center")
    plt.yticks(np.arange(k) + .5, labels, fontsize=8)
    plt.xlabel("Weight")
    plt.ylabel("Position")
    plt.savefig("result/H3K4me1/top50_weighted_position_" + str(c) + ".pdf", dpi = 300)
    plt.close() 

In [ ]:
exp_df.to_csv('result/H3K4me1/top50_weighted_position_level.tsv', sep = '\t')

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(exp_df, annot=False, cmap='coolwarm', yticklabels=False)
plt.savefig("result/H3K4me1/top50_weighted_position_level_heatmap.pdf", dpi = 300, bbox_inches='tight')
plt.close()

# H3K4me1-test

In [ ]:
data = pd.read_table('../matrix/H3K4me1_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)

In [ ]:
data.columns = data.columns.str[:-5]
data_avg = data.groupby(data.columns, axis=1).mean()

In [ ]:
data = np.array(data_avg.T)

In [ ]:
# level of sparsity
print(np.sum(data != 0)/data.size)

# negative elements
print(np.sum(data < 0))

In [ ]:
# normalize data
data = preprocessing.Normalizer().fit_transform(data)

## Clustering of genes

In [ ]:
rank = 10
nmf = nimfa.Nmf(data, rank=rank, seed='random_vcol', max_iter=100, 
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

### plot dendrogram along with the heatmap

In [ ]:
def clean_axis(ax):
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C = 1 - nmf_fit.fit.consensus()
Y = sch.linkage(C, method='average')

denAX = fig.add_subplot(heatmapGS[0,0])
denD = sch.dendrogram(Y, orientation='right', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D = C[denD['leaves'], :][:, denD['leaves']]
axi = heatmapAX.imshow(D, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

# H3K9me3

In [ ]:
data = pd.read_table('../matrix/H3K9me3_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)

In [ ]:
data.columns = data.columns.str[:-5]
data_avg = data.groupby(data.columns, axis=1).mean()

In [ ]:
data = np.array(data_avg.T)

In [ ]:
# level of sparsity
print(np.sum(data != 0)/data.size)

# negative elements
print(np.sum(data < 0))

In [ ]:
# normalize data
data = preprocessing.Normalizer().fit_transform(data)

## Clustering of genes

In [ ]:
rank = 10
nmf = nimfa.Nmf(data, rank=rank, seed='random_vcol', max_iter=100, 
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

### plot dendrogram along with the heatmap

In [ ]:
def clean_axis(ax):
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C = 1 - nmf_fit.fit.consensus()
Y = sch.linkage(C, method='average')

denAX = fig.add_subplot(heatmapGS[0,0])
denD = sch.dendrogram(Y, orientation='right', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D = C[denD['leaves'], :][:, denD['leaves']]
axi = heatmapAX.imshow(D, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

# H3K4me3

In [ ]:
data = pd.read_table('../matrix/H3K4me3_variable_peak_matrix.tsv',
                     sep = '\t', header=0, index_col=0)

In [ ]:
data.columns = data.columns.str[:-5]
data_avg = data.groupby(data.columns, axis=1).mean()

In [ ]:
data = np.array(data_avg.T)

In [ ]:
# level of sparsity
print(np.sum(data != 0)/data.size)

# negative elements
print(np.sum(data < 0))

In [ ]:
# normalize data
data = preprocessing.Normalizer().fit_transform(data)

## Clustering of genes

In [ ]:
rank = 10
nmf = nimfa.Nmf(data, rank=rank, seed='random_vcol', max_iter=100, 
                update='divergence', objective='div', n_run=50, track_factor=True)
nmf_fit = nmf()

### plot dendrogram along with the heatmap

In [ ]:
import time

In [ ]:
def clean_axis(ax):
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C = 1 - nmf_fit.fit.consensus()
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) + ' - start calculating linkage!')
Y = sch.linkage(C, method='average')
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) +  'end calculating linkage!')



In [ ]:
denD['leaves']

In [ ]:
import sys
sys.setrecursionlimit(340000000)

In [ ]:
fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

denAX = fig.add_subplot(heatmapGS[0,0])
denD = sch.dendrogram(Y, orientation='right', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D = C[denD['leaves'], :][:, denD['leaves']]
axi = heatmapAX.imshow(D, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

### test on manual setting W and H

In [ ]:
# get reproducable result

nmf_test = nimfa.Nmf(V=data, rank=10, seed='fixed',W=nmf_fit.fit.W, H=nmf_fit.fit.H, max_iter=0, 
                update='divergence', objective='div', n_run=0, track_factor=True)
nmf_fit_test = nmf()

In [ ]:
fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C_test = 1 - nmf_fit_test.fit.consensus()
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) + ' - start calculating linkage!')
Y_test = sch.linkage(C_test, method='average')
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) +  'end calculating linkage!')

In [ ]:
fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

denAX = fig.add_subplot(heatmapGS[0,0])
denD_test = sch.dendrogram(Y_test, orientation='right', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D_test = C_test[denD_test['leaves'], :][:, denD_test['leaves']]
axi = heatmapAX.imshow(D_test, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

In [ ]:
plt.figure(dpi = 120, figsize = (5,5))
plt.subplot(1,2,1)
plt.imshow(nmf_fit.fit.W, aspect='auto')
plt.title('W')
plt.subplot(1,2,2)
plt.imshow(nmf_fit.fit.H, aspect='auto')
plt.title('H')

### test on Nndsvd initial from issue

In [ ]:
# get reproducable result
from nimfa.methods.seeding.nndsvd import Nndsvd

s = Nndsvd()
w,h = s.initialize(data, rank = 10)
nmf_nndsvd = nimfa.Nmf(V=data, rank=2, W=w, H=h)
nmf_fit_nndsvd = nmf_nndsvd()

In [ ]:
fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

C_nndsvd = 1 - nmf_fit_nndsvd.fit.consensus()
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) + ' - start calculating linkage!')
Y_nndsvd = sch.linkage(C_nndsvd, method='average')
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()) +  'end calculating linkage!')

In [ ]:
fig = plt.figure(figsize=(13.9, 10))
heatmapGS = gridspec.GridSpec(1, 2, wspace=.01, hspace=0., width_ratios=[0.25,1])

denAX = fig.add_subplot(heatmapGS[0,0])
denD_nndsvd = sch.dendrogram(Y_nndsvd, orientation='right', link_color_func=lambda k: 'black')
clean_axis(denAX)

heatmapAX = fig.add_subplot(heatmapGS[0,1])
D_nndsvd = C_test[denD_nndsvd['leaves'], :][:, denD_nndsvd['leaves']]
axi = heatmapAX.imshow(D_nndsvd, interpolation='nearest', aspect='equal', origin='lower', cmap='RdBu') 
clean_axis(heatmapAX)

cb = fig.colorbar(axi, fraction=0.046, pad=0.04, aspect=10) 
cb.set_label('Distance', fontsize=20)

In [ ]:
plt.figure(dpi = 120, figsize = (5,5))
plt.subplot(1,2,1)
plt.imshow(nmf_fit.fit.W, aspect='auto')
plt.title('W')
plt.subplot(1,2,2)
plt.imshow(nmf_fit.fit.H, aspect='auto')
plt.title('H')

### Latent dimensionality selection

In [ ]:
rank_cands = range(3, 30, 5)
snmf = nimfa.Snmf(data, seed='random_vcol', max_iter=100)
summary = snmf.estimate_rank(rank_range=rank_cands, n_run=10, what='all')

In [ ]:
rss = [summary[rank]['rss'] for rank in rank_cands]
coph = [summary[rank]['cophenetic'] for rank in rank_cands]
disp = [summary[rank]['dispersion'] for rank in rank_cands]
spar = [summary[rank]['sparseness'] for rank in rank_cands]
spar_w, spar_h = zip(*spar)
evar = [summary[rank]['evar'] for rank in rank_cands]

plt.plot(rank_cands, rss, 'o-', label='RSS', linewidth=2)
plt.plot(rank_cands, coph, 'o-', label='Cophenetic correlation', linewidth=2)
plt.plot(rank_cands, disp,'o-', label='Dispersion', linewidth=2)
plt.plot(rank_cands, spar_w, 'o-', label='Sparsity (Basis)', linewidth=2)
plt.plot(rank_cands, spar_h, 'o-', label='Sparsity (Mixture)', linewidth=2)
plt.plot(rank_cands, evar, 'o-', label='Explained variance', linewidth=2)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, numpoints=1);

In [ ]:
# SNMF/L: for sparse W (where ‘l’ denotes the sparseness imposed on the left factor)
# SNMF/R: for sparse H (where ‘r’ denotes the sparseness imposed on the right factor)
rank = 8
snmf = nimfa.Snmf(data.T, seed='random_vcol', version='l', max_iter=200)
snmf_fit = snmf()

In [ ]:
W = snmf_fit.fit.basis()

c = 0
k = 10

topk = np.argsort(np.asarray(W[:, c]).flatten())[-k:]
val = W[topk, c]

plt.barh(np.arange(k) + .5, np.array(val).flatten(), align="center")
labels = [id2gene[idx] for idx in topk]
plt.yticks(np.arange(k) + .5, labels)
plt.xlabel("Weight")
plt.ylabel("Gene");